# 04 · Heart-rate track: pulse-lost / dangerous-rhythm detector from PPG

Can the PPG waveform alone tell a real dangerous rhythm, where the pulse disappears or turns chaotic, from a false monitor alarm? And how often would it false-alarm on a healthy chest? Heart-rate trends have no usable signature for hypoxic drowning, so this track targets cardiac events. The notebook contains **no drowning or loss-of-consciousness labels** and cannot establish detection performance for those events. All outputs are research-only (`deployable: false`).

| | |
|---|---|
| **Training and test data** | [PhysioNet/CinC Challenge 2015](https://physionet.org/content/challenge-2015/1.0.0/) (open, ODC-BY 1.0): 750 ICU alarm records, each ending at the moment a bedside monitor raised an alarm. **Used:** asystole, extreme bradycardia, ventricular tachycardia and ventricular flutter/fibrillation records **that include PPG** (514 records, 123 true). **Excluded:** extreme tachycardia, because a heart rate above 140 is normal while swimming, and records without PPG, because the wearable has only PPG. |
| **Labels** | **Expert annotators judged each alarm true or false**; a record was kept only when at least two-thirds agreed. These are the investigators' labels; the notebook only selects records and never relabels. |
| **Input** | The 16 s of finger PPG before the alarm, resampled from 250 Hz to the wearable's **100 Hz**. The event starts between 4:50 and 5:00, so the window contains it. |
| **False-alarm data** | [WildPPG](https://siplab.org/projects/WildPPG): healthy volunteers in daily life, **sternum** PPG at 128 Hz, the same chest site as the wearable. Every alert there counts as a false alarm; that is a protocol fact (healthy volunteers), not a derived label. Eight participants provide healthy negatives and fit the gate; the other eight are read only after every model and threshold is frozen. |
| **Outputs** | Configuration, cross-validation and false-alarm tables, figures, models and the firmware-selection record in `outputs/pulse_loss/`. |

**Models compared** (ESP32 execution is not measured here)

| Name | Type | Trained? |
|---|---|---|
| `rule_definitions` | No pulse for ≥ 4 s **or** 5-beat heart rate < 40 bpm (the Challenge's own asystole and bradycardia definitions, applied to PPG beats) | No: published thresholds, nothing tuned |
| `logistic` | Logistic regression on 10 PPG features | Yes |
| `boosted_trees` | Gradient-boosted trees on the same features | Yes |
| `cnn_1d` | Small 1D CNN on the raw 16-s waveform (about 3,600 parameters) | Yes; skipped if PyTorch is unavailable |
| `boosted_trees_h` | `boosted_trees`, plus healthy chest windows as extra negatives | Yes |
| `cnn_v5`, `cnn_v5_h` | 3-seed CNN ensemble with a ≈ 5-s receptive field, eight 2-s time bins, early stopping and signal-corruption augmentation; `_h` adds healthy negatives | Yes; skipped if PyTorch is unavailable |

Two deployment options are evaluated on top of every model: a **pulse-visibility gate** (PPG only) decided on the window *before* the one being scored, and **2-of-3 persistence**.

> **Limits.**
> - It cannot support detection of drowning, or performance on the wearable's own chest MAX30102 signal in water.
> - Folds are record-level. The Challenge does not publish patient IDs, so a patient may appear in more than one fold.
> - All 16 WildPPG people appear in saved notebook 02 results. The 8/8 split is person-disjoint within this run but is not untouched validation. The run-plan hash is an audit record, not a public preregistration, and the inherited selection rule is a provisional ranking only.
> - The gate learns an automatic ECG/PPG beat-count agreement. Equal counts do not prove matched beats or a visible pulse.
>
> What v5 changed from v4, and the ECG-counter correction, are in the [folder README](README.md#version-history).

## How to run

On Google Colab, use a CPU runtime and choose **Runtime → Run all**. Challenge 2015 (339 MB) is cached on Drive. WildPPG uses notebook 02's verified downloader and cache: about 300 MB per participant, 16 participants. The full local run took about 113 minutes.

| Setting | Effect |
|---|---|
| `PULSE_QUICK=1` | Smoke test (short recordings, few epochs). Use a separate output directory. A smoke test never selects a model and must not be reported. |
| `PULSE_DATA_ROOT`, `PULSE_CACHE`, `PULSE_OUTPUT_DIR` | Folder with attached originals, download cache and outputs |
| `PULSE_DOWNLOAD=0` | Never download; use attached or cached files only |
| `WILDPPG_CACHE` | WildPPG cache shared with notebook 02 |
| `PULSE_JOBS` | Parallel WildPPG workers; each holds one ≈ 12-h recording (≈ 1 GB) in memory |
| `KB_SAVE_TO_DRIVE=0` | Keep everything in the runtime instead of Drive |

The full run uses its fixed 8/8 participant split; `WILDPPG_IDS` does not override it.

**Checkpoint cache.** Each fit is cached under a key that includes `AUDITED_CODE_SHA256`. That is an explicit version key, not an automatic fingerprint of edited cells: after changing training or preprocessing code, use a new cache or change the key. Joblib files are loaded only from this notebook's own trusted local cache.

## Setup

Package check and imports, then storage, the experiment constants, the run plan and the confidence-interval helpers.

In [ ]:
# Version key for the fit-checkpoint cache. Not computed from the code: change it after editing
# training code.
AUDITED_CODE_SHA256 = "e0edfe9d9595b99043d7770c5ee7d9ed16f2898151d72f7edc1643994666ba71"
import importlib.util

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
    "requests": "requests",
    "matplotlib": "matplotlib",
    "IPython": "ipython",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise ImportError(
        "Missing packages: "
        + ", ".join(missing)
        + ". Install only these with %pip, restart and Run All."
    )
from pathlib import Path
import hashlib, io, json, math, os, platform, sys, time, zipfile
import numpy as np
import pandas as pd
import requests, joblib, sklearn
from scipy import io as sio, signal, stats
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from IPython.display import display

HAVE_TORCH = importlib.util.find_spec("torch") is not None


def env_flag(name, default):
    return os.getenv(name, str(default)).lower() in {"1", "true", "yes"}

### Storage

In [ ]:
# ---- Storage: Google Colab ----
ON_COLAB = "google.colab" in sys.modules
SAVE_TO_DRIVE = env_flag("KB_SAVE_TO_DRIVE", ON_COLAB)
if ON_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/KappaBetaML")
else:
    BASE = Path("/content") if ON_COLAB else Path(".")
DATA_ROOT = Path(os.getenv("PULSE_DATA_ROOT", str(BASE / "input")))
OUT = Path(os.getenv("PULSE_OUTPUT_DIR", str(BASE / "outputs" / "pulse_loss")))
FIGURES = OUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
CACHE = Path(os.getenv("PULSE_CACHE", str(BASE / "cache" / "pulse_loss")))
# Same default cache as notebook 02, so WildPPG is downloaded once per runtime.
WILDPPG_CACHE = Path(
    os.getenv(
        "WILDPPG_CACHE",
        "/content/cache/wildppg_sternum" if ON_COLAB else str(BASE / "cache" / "wildppg_sternum"),
    )
)
# None = all 16
WILDPPG_IDS = [s.strip() for s in os.getenv("WILDPPG_IDS", "").split(",") if s.strip()] or None
ALLOW_DOWNLOAD = env_flag("PULSE_DOWNLOAD", True)
SEED = 20261109

### Experiment constants

Everything here is fixed before results are seen.

In [ ]:
# ---- ICU experiment: fixed in v4, before any result was seen. ----
C2015_FS, DEVICE_FS = 250, 100  # Challenge sampling rate; wearable raw PPG rate
WINDOW_S, ALARM_AT_S = 16, 300  # 16 s of PPG ending at the alarm (event onset 4:50-5:00)
TARGET_ALARMS = ["Asystole", "Bradycardia", "Ventricular_Tachycardia", "Ventricular_Flutter_Fib"]
EXCLUDED_ALARMS = {"Tachycardia": "HR > 140 bpm is normal while swimming"}
# Challenge alarm definitions (rule_definitions)
ASYSTOLE_GAP_S, BRADY_BPM, BRADY_BEATS = 4.0, 40.0, 5
OUTER_FOLDS, INNER_FOLDS = 5, 5
# Two operating points, both declared in advance and chosen on inner (training-only) predictions:
TARGET_SENSITIVITY = 0.90  # 'high_sensitivity': largest threshold with inner sensitivity >= 0.90
TARGET_SPECIFICITY = 0.90  # 'high_specificity': smallest threshold with inner specificity >= 0.90
TRANSFER_STEP_S = 2  # healthy chest PPG scored every 2 s (16-s windows)

# ---- Healthy chest PPG, gate, augmentation and cnn_v5 (v5, 2026-11-16). ----
# Fixed here and hashed into v5_run_plan.json before any v5 result.
# smoke test only (short recordings, few epochs); never report it
QUICK = env_flag("PULSE_QUICK", False)
# scored in v4: training + gate only
WILDPPG_TRAIN_IDS = ["an0", "e61", "fex", "k2s", "kjd", "l38", "n31", "ngh"]
# held out from this fit; already inspected in notebook 02
WILDPPG_TEST_IDS = ["p5d", "p9p", "qm9", "ssx", "trh", "tz8", "u7y", "w4p"]
if QUICK:
    WILDPPG_TRAIN_IDS, WILDPPG_TEST_IDS = WILDPPG_TRAIN_IDS[:2], WILDPPG_TEST_IDS[:2]
WILDPPG_IDS = WILDPPG_TRAIN_IDS + WILDPPG_TEST_IDS
WILDPPG_MAX_S = 1800 if QUICK else None  # full recordings (~12 h each) unless smoke-testing
TRAIN_STEP_S = 16  # non-overlapping windows from training participants
# pulse-visible windows per training participant used as negatives
HEALTHY_NEG_PER_PARTICIPANT = 50
# PPG beats / ECG beats counted as 'pulse visible' (v4 definition)
VISIBLE_RATIO = (0.8, 1.2)
# gate threshold: lowest with leave-one-participant-out precision >= 0.90
GATE_MIN_PRECISION = 0.90
# secondary: alert only if >= 2 of the last 3 windows (4 s) flag
PERSIST_K, PERSIST_N = 2, 3
# augmented copies per ICU record; chance a record is swapped per epoch
AUG_COPIES, AUG_PROB = 4, 0.5
CNN4_EPOCHS = 2 if QUICK else 40  # v4 CNN, unchanged
CNN5_SEEDS, CNN5_MAX_EPOCHS, CNN5_PATIENCE, CNN5_VALID_FRACTION = (
    (1, 4, 2, 0.15) if QUICK else (3, 80, 10, 0.15)
)

### Pre-declared selection rule and run plan

`SELECTION_RULE` states, in order:
1. Candidates are `rule_definitions` and each learned model at its `high_specificity` point.
2. Keep those whose ICU outer-CV sensitivity is ≥ that of `rule_definitions` (untuned published definitions).
3. Choose the one with the fewest false alert starts per monitored hour on `WILDPPG_TEST_IDS` under the prior-window gate.
4. Break ties by AUROC. 2-of-3 persistence is reported, not used for selection.

All constants are written to `v5_run_plan.json` and hashed. Re-running into an output folder that holds a different plan stops the run.

In [ ]:
SELECTION_RULE = (
    "Among rule_definitions and each learned model at its high_specificity point: keep those whose ICU "
    "outer-CV sensitivity is >= that of rule_definitions (untuned published definitions); choose the one "
    "with the fewest false alert starts per monitored hour on WILDPPG_TEST_IDS under the prior-window "
    "gate; break ties by AUROC. 2-of-3 persistence is reported, not used for selection."
)
PREREG = {
    k: v
    for k, v in dict(
        version="v5 audit 2026-11-16",
        inspected_split=True,
        independent_validation=False,
        seed=SEED,
        wildppg_train=WILDPPG_TRAIN_IDS,
        wildppg_test=WILDPPG_TEST_IDS,
        train_step_s=TRAIN_STEP_S,
        healthy_neg_per_participant=HEALTHY_NEG_PER_PARTICIPANT,
        visible_ratio=VISIBLE_RATIO,
        gate_min_precision=GATE_MIN_PRECISION,
        persistence=[PERSIST_K, PERSIST_N],
        augmentation={
            "copies": AUG_COPIES,
            "swap_probability": AUG_PROB,
            "kinds": [
                "respiratory amplitude modulation <= 30%",
                "healthy-chest motion artefact burst 2-6 s",
                "band-limited sensor noise",
            ],
            "excluded": "time stretching (changes heart rate)",
        },
        cnn_v5={
            "seeds": CNN5_SEEDS,
            "max_epochs": CNN5_MAX_EPOCHS,
            "patience": CNN5_PATIENCE,
            "early_stop_fraction": CNN5_VALID_FRACTION,
            "optimizer": "AdamW lr 2e-3 wd 1e-3",
            "batch": 32,
        },
        gate_model="HistGradientBoosting, same fixed hyperparameters as boosted_trees",
        selection_rule=SELECTION_RULE,
        quick=QUICK,
    ).items()
}
_prereg = json.dumps(PREREG, indent=2, sort_keys=True)
_plan_path = OUT / "v5_run_plan.json"
if _plan_path.exists() and _plan_path.read_text(encoding="utf-8") != _prereg:
    raise ValueError("Run plan changed: use a new output directory to preserve provenance")
_plan_path.write_text(_prereg, encoding="utf-8")
print(
    "v5 development run-plan sha256:",
    hashlib.sha256(_prereg.encode()).hexdigest(),
    "| QUICK" if QUICK else "",
)

### Confidence intervals

In [ ]:
def poisson_rate_ci(count, hours, level=0.95):
    a = 1 - level
    lo = 0.0 if count == 0 else stats.chi2.ppf(a / 2, 2 * count) / 2
    return lo / hours, stats.chi2.ppf(1 - a / 2, 2 * count + 2) / 2 / hours


def proportion_ci(k, n, level=0.95):
    if n == 0:
        return np.nan, np.nan
    ci = stats.binomtest(int(k), int(n)).proportion_ci(confidence_level=level, method="wilson")
    return ci.low, ci.high


print("Output:", OUT.resolve(), "| PyTorch available:", HAVE_TORCH)

## ICU training data: acquire and verify

The zip is pinned by MD5 (`99a51fefcd72b584a8122ebd7f78dacf`, 338,623,761 bytes, checked 2026-11-09). A mismatch stops the run.

In [ ]:
C2015 = (
    "training.zip",
    "https://physionet.org/files/challenge-2015/1.0.0/training.zip",
    "99a51fefcd72b584a8122ebd7f78dacf",
)


def md5(path, chunk=1 << 22):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def acquire(name, url, digest):
    found = list(DATA_ROOT.rglob(name)) if DATA_ROOT.is_dir() else []
    for path in [*found, CACHE / name]:
        if path.is_file() and md5(path) == digest:
            return path
    if not ALLOW_DOWNLOAD:
        raise FileNotFoundError(f"Attach the original {name} below {DATA_ROOT} or enable Internet.")
    CACHE.mkdir(parents=True, exist_ok=True)
    part = CACHE / (name + ".part")
    for attempt in range(4):
        try:
            offset = part.stat().st_size if part.exists() else 0
            headers = {
                "User-Agent": "pulse-loss-notebook/1.0",
                **({"Range": f"bytes={offset}-"} if offset else {}),
            }
            with requests.get(url, stream=True, timeout=(15, 120), headers=headers) as r:
                r.raise_for_status()
                with open(part, "ab" if offset and r.status_code == 206 else "wb") as f:
                    for block in r.iter_content(1 << 20):
                        f.write(block)
            if md5(part) != digest:
                part.unlink()
                raise ValueError("Checksum mismatch; the source changed. Review before use.")
            part.replace(CACHE / name)
            return CACHE / name
        except (requests.RequestException, OSError, ValueError) as error:
            last = error
            time.sleep(2 + 2 * attempt)
    raise RuntimeError(f"Cannot acquire {name}: {last}")


c2015_zip = acquire(*C2015)
print(c2015_zip, f"{c2015_zip.stat().st_size / 1e6:.1f} MB")

## Labels and PPG windows

Each record's label comes from the Challenge's `ALARMS` file (`record,alarm_type,label`; `1` = true alarm, `0` = false alarm), as assigned by the expert annotators. We only select which records to use; we never relabel.

| Alarm | Challenge definition | Used? |
|---|---|---|
| Asystole | No QRS for ≥ 4 s | Yes |
| Extreme bradycardia | HR < 40 bpm for 5 beats | Yes |
| Ventricular tachycardia | ≥ 5 ventricular beats at HR > 100 | Yes |
| Ventricular flutter/fibrillation | Fibrillatory waveform for ≥ 4 s | Yes |
| Extreme tachycardia | HR > 140 bpm for 17 beats | **No**: normal while swimming |

**PPG preparation** (identical for every dataset):
1. Convert to physical units with the header gain.
2. Resample to 100 Hz.
3. Band-pass 0.5–8 Hz.
4. Scale by the window's 5th–95th percentile spread.

The first 4 min 50 s of each record is **not used as "normal"**: it is unlabelled.

In [ ]:
def read_header(z, record):
    lines = z.read(f"training/{record}.hea").decode("latin1").splitlines()
    sigs = []
    for line in lines[1:]:
        if line and not line.startswith("#"):
            f = line.split()
            sigs.append((f[-1], float(f[2].split("/")[0].split("(")[0])))
    return sigs


def to_device_rate(x, fs):
    up, down = DEVICE_FS, fs
    g = math.gcd(up, down)
    return signal.resample_poly(np.asarray(x, float), up // g, down // g)


windows_path = CACHE / "c2015_ppg_windows.npz"
if windows_path.exists():
    cached = np.load(windows_path, allow_pickle=False)
    X_wave, meta = cached["X"], pd.DataFrame({k: cached[k] for k in ["record", "alarm", "label"]})
else:
    with zipfile.ZipFile(c2015_zip) as z:
        alarms = pd.read_csv(
            io.BytesIO(z.read("training/ALARMS")), header=None, names=["record", "alarm", "label"]
        )
        waves, rows = [], []
        for r in alarms.itertuples():
            if r.alarm not in TARGET_ALARMS:
                continue
            sigs = read_header(z, r.record)
            names = [n for n, _ in sigs]
            if "PLETH" not in names:
                continue
            k = names.index("PLETH")
            val = (
                sio.loadmat(io.BytesIO(z.read(f"training/{r.record}.mat")))["val"][k].astype(float)
                / sigs[k][1]
            )
            seg = val[(ALARM_AT_S - WINDOW_S) * C2015_FS : ALARM_AT_S * C2015_FS]
            waves.append(to_device_rate(seg, C2015_FS)[: WINDOW_S * DEVICE_FS])
            rows.append({"record": r.record, "alarm": r.alarm, "label": int(r.label)})
    X_wave, meta = np.vstack(waves).astype(np.float32), pd.DataFrame(rows)
    CACHE.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        windows_path,
        X=X_wave,
        record=meta.record.to_numpy(dtype=str),
        alarm=meta.alarm.to_numpy(dtype=str),
        label=meta.label.to_numpy(dtype=int),
    )
y = meta.label.to_numpy().astype(int)
display(
    meta.groupby("alarm")
    .label.agg(records="size", true_alarms="sum")
    .assign(false_alarms=lambda d: d.records - d.true_alarms)
)
print(
    f"{len(meta)} records with PPG; {y.sum()} expert-confirmed true events; window {X_wave.shape[1] / DEVICE_FS:.0f} s at {DEVICE_FS} Hz"
)

### Extra windows for the timing analyses

Windows ending 4 s and 2 s before the alarm (for 2-of-3 persistence), and the window from 268 to 284 s (what a prior-window gate would see).

In [ ]:
# Three more 16-s windows per record, each resampled on its own exactly like X_wave and cached
# separately: ending 4 s and 2 s before the alarm (2-of-3 persistence) and 268-284 s (what a
# prior-window gate would see).
EXTRA_ENDS = {"m4": ALARM_AT_S - 4, "m2": ALARM_AT_S - 2, "pre": ALARM_AT_S - WINDOW_S}
extra_path = CACHE / "c2015_ppg_windows_v5_extra.npz"
if extra_path.exists():
    cached = np.load(extra_path, allow_pickle=False)
    assert (
        cached["record"] == meta.record.to_numpy(dtype=str)
    ).all(), "extra-window cache does not match records"
    X_extra = {k: cached[k] for k in EXTRA_ENDS}
else:
    X_extra = {k: [] for k in EXTRA_ENDS}
    with zipfile.ZipFile(c2015_zip) as z:
        for record in meta.record:
            sigs = read_header(z, record)
            k_pleth = [n for n, _ in sigs].index("PLETH")
            val = (
                sio.loadmat(io.BytesIO(z.read(f"training/{record}.mat")))["val"][k_pleth].astype(
                    float
                )
                / sigs[k_pleth][1]
            )
            for key, end in EXTRA_ENDS.items():
                X_extra[key].append(
                    to_device_rate(val[(end - WINDOW_S) * C2015_FS : end * C2015_FS], C2015_FS)[
                        : WINDOW_S * DEVICE_FS
                    ]
                )
    X_extra = {k: np.vstack(v).astype(np.float32) for k, v in X_extra.items()}
    np.savez_compressed(extra_path, record=meta.record.to_numpy(dtype=str), **X_extra)
print("extra ICU windows:", {k: v.shape for k, v in X_extra.items()})

## Features and architectures

**PPG features** (one value per 16-s window, from detected pulse peaks):

| Feature | Meaning |
|---|---|
| `max_gap_s` | Longest time without a detected pulse, including the window edges |
| `tail_gap_s` | Time since the last pulse at the end of the window |
| `n_beats` | Number of pulses |
| `hr_median`, `hr_min5` | Median heart rate; lowest 5-beat average heart rate |
| `ibi_cv` | Irregularity of pulse intervals |
| `amp_tail_ratio` | Pulse size in the last 4 s ÷ pulse size in the first 8 s (pulse fading) |
| `spectral_purity` | Share of 0.5–4 Hz power near the dominant pulse frequency (low = noise or chaos) |
| `flat_fraction` | Share of near-flat samples (signal lost) |
| `kurtosis` | Spikiness (motion artefacts) |

`cnn_1d` sees the prepared waveform directly. Three convolution blocks (8, 16, 32 channels) feed global averaging and one output, about 3,600 parameters. It is trained with class weighting and fixed seeds.

In [ ]:
SOS = signal.butter(3, [0.5, 8.0], fs=DEVICE_FS, btype="band", output="sos")
FEATURES = [
    "max_gap_s",
    "tail_gap_s",
    "n_beats",
    "hr_median",
    "hr_min5",
    "ibi_cv",
    "amp_tail_ratio",
    "spectral_purity",
    "flat_fraction",
    "kurtosis",
]


def prepare(x):
    x = signal.sosfiltfilt(SOS, np.asarray(x, float) - np.median(x))
    spread = np.percentile(x, 95) - np.percentile(x, 5)
    return x / spread if spread > 0 else np.zeros_like(x)


def ppg_features(raw):
    z = prepare(raw)
    duration = len(z) / DEVICE_FS
    if not np.any(z):
        return dict(
            zip(FEATURES, [duration, duration, 0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0])
        ), np.array([], int)
    peaks, prop = signal.find_peaks(z, distance=int(0.3 * DEVICE_FS), prominence=0.3)
    t = peaks / DEVICE_FS
    gaps = np.diff(np.r_[0.0, t, duration])
    ibi = np.diff(t)
    hr = 60 / ibi if len(ibi) else np.array([0.0])
    hr5 = (
        60 / pd.Series(ibi).rolling(BRADY_BEATS).mean().dropna().to_numpy()
        if len(ibi) >= BRADY_BEATS
        else hr
    )
    amps = prop["prominences"]
    tail, head = amps[t > duration - 4], amps[t < 8]
    f, p = signal.welch(z, fs=DEVICE_FS, nperseg=min(len(z), 512))
    band = (f >= 0.5) & (f <= 4)
    peak_f = f[band][np.argmax(p[band])]
    purity = p[band][np.abs(f[band] - peak_f) <= 0.25].sum() / max(p[band].sum(), 1e-12)
    values = [
        gaps.max(),
        gaps[-1],
        len(peaks),
        float(np.median(hr)),
        float(np.min(hr5)),
        float(np.std(ibi) / np.mean(ibi)) if len(ibi) > 1 else 0.0,
        float(np.median(tail) / np.median(head)) if len(tail) and len(head) else 0.0,
        float(purity),
        float((np.abs(np.diff(z)) < 0.002).mean()),
        float(stats.kurtosis(z)),
    ]
    return dict(zip(FEATURES, values)), peaks


def rule_definitions(F):
    # Challenge alarm definitions applied to PPG pulses: no pulse >= 4 s, or 5-beat HR < 40 bpm.
    return (
        ((F["max_gap_s"] >= ASYSTOLE_GAP_S) | ((F["hr_min5"] < BRADY_BPM) & (F["n_beats"] > 0)))
        .astype(float)
        .to_numpy()
    )

Compute the features and the normalised waveform for every ICU record.

In [ ]:
feature_table = pd.DataFrame([ppg_features(w)[0] for w in X_wave])
X_feat = feature_table[FEATURES].to_numpy(float)
X_norm = np.vstack([prepare(w) for w in X_wave]).astype(np.float32)

### Losses and the feature models

Every learned model minimises a class-weighted log loss. `balanced_log_loss` reports the same quantity, so train, validation and test values are on the training objective's scale.

In [ ]:
def balanced_log_loss(labels, p):
    # Mean of the per-class mean log losses. Every learned model here is trained on a class-weighted
    # log loss (class_weight='balanced' or pos_weight = negatives/positives), which is proportional
    # to this quantity, so train, validation and test values are on the same scale as the training
    # objective.
    labels, p = np.asarray(labels), np.clip(np.asarray(p, float), 1e-7, 1 - 1e-7)
    parts = [
        -np.log(p[labels == 1]).mean() if (labels == 1).any() else np.nan,
        -np.log(1 - p[labels == 0]).mean() if (labels == 0).any() else np.nan,
    ]
    return float(np.nanmean(parts))


def split_scores(labels, p, threshold=0.5):
    # Loss plus accuracy at 0.5, the natural decision point of a class-balanced model (not an
    # operating point).
    labels, flag = np.asarray(labels), np.asarray(p) >= threshold
    se = flag[labels == 1].mean() if (labels == 1).any() else np.nan
    sp = (~flag[labels == 0]).mean() if (labels == 0).any() else np.nan
    return {
        "loss": balanced_log_loss(labels, p),
        "accuracy": float((flag == (labels == 1)).mean()),
        "balanced_accuracy": float(np.nanmean([se, sp])),
    }


def make_feature_model(name):
    if name == "logistic":
        return make_pipeline(
            StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")
        )
    return HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        min_samples_leaf=20,
        l2_regularization=1.0,
        class_weight="balanced",
        random_state=SEED,
    )

### `cnn_1d`: the original small CNN

In [ ]:
if HAVE_TORCH:
    import torch
    from torch import nn

    torch.set_num_threads(min(4, os.cpu_count() or 1))

    class PulseCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.body = nn.Sequential(
                nn.Conv1d(1, 8, 7, stride=2, padding=3),
                nn.ReLU(),
                nn.MaxPool1d(2),
                nn.Conv1d(8, 16, 7, stride=2, padding=3),
                nn.ReLU(),
                nn.MaxPool1d(2),
                nn.Conv1d(16, 32, 5, padding=2),
                nn.ReLU(),
                nn.AdaptiveAvgPool1d(1),
            )
            self.head = nn.Linear(32, 1)

        def forward(self, x):
            return self.head(self.body(x.unsqueeze(1)).squeeze(-1)).squeeze(-1)

    class CNNModel:
        def __init__(self, epochs=None, seed=SEED):
            self.epochs, self.seed = CNN4_EPOCHS if epochs is None else epochs, seed

        def fit(self, X, y, monitor=None):
            # monitor: optional {split: (X, y)} scored after every epoch. Scoring uses no randomness
            # and no gradients, so the trained weights are identical with or without it.
            torch.manual_seed(self.seed)
            rng = np.random.default_rng(self.seed)
            self.net = PulseCNN()
            self.history = []
            opt = torch.optim.Adam(self.net.parameters(), lr=1e-3, weight_decay=1e-3)
            pos = max(1, int(y.sum()))
            loss_fn = nn.BCEWithLogitsLoss(
                pos_weight=torch.tensor((len(y) - pos) / pos, dtype=torch.float32)
            )
            Xt, yt = torch.tensor(X), torch.tensor(y, dtype=torch.float32)
            for epoch in range(self.epochs):
                self.net.train()
                order = rng.permutation(len(y))
                for b in range(0, len(order), 32):
                    idx = order[b : b + 32]
                    # amplitude jitter
                    gain = torch.tensor(rng.uniform(0.8, 1.2, (len(idx), 1)), dtype=torch.float32)
                    opt.zero_grad()
                    loss_fn(self.net(Xt[idx] * gain), yt[idx]).backward()
                    opt.step()
                for split, (Xm, ym) in (monitor or {}).items():
                    self.history.append(
                        {
                            "step": epoch + 1,
                            "split": split,
                            **split_scores(ym, self.predict_proba(Xm)[:, 1]),
                        }
                    )
            return self

        def predict_proba(self, X):
            self.net.eval()
            with torch.no_grad():
                p = np.concatenate(
                    [
                        torch.sigmoid(
                            self.net(torch.tensor(np.asarray(X[i : i + 256], np.float32)))
                        ).numpy()
                        for i in range(0, len(X), 256)
                    ]
                )
            return np.c_[1 - p, p]

    print("cnn_1d parameters:", sum(p.numel() for p in PulseCNN().parameters()))

## Healthy chest PPG: negatives and the pulse-visibility gate

The eight comparison participants are excluded from this run's model/gate fitting and threshold choice. All 16 were inspected in earlier project results; this is development data. The gate learns an automatic instrument-agreement proxy.

### WildPPG downloader

Identical to notebook 02: fetches and verifies the sternum portion of each original file.

In [ ]:
# The original MATLAB v5 file begins with id, notes and the complete sternum matrix.
# Each manifest entry pins the byte immediately after that matrix and the source ETag.
# Downloading this exact prefix preserves every chest sample at its original precision.
from pathlib import Path
import shutil
import time
from urllib.error import URLError
from urllib.request import Request, urlopen
from scipy.io import whosmat

WILDPPG_BASE = "https://polybox.ethz.ch/public.php/dav/files/NWTuyNojU7aya1y/data/"
STERNUM_FILES = {
    "an0": (295671696, "d185bc1ae93e4c9848524e1aab22a39b"),
    "e61": (296257912, "f88e7a783153615a128013363f2d548e"),
    "fex": (320834792, "7273ac5694d4dd2f91aa264e00ddf00a"),
    "k2s": (365403792, "52134984b7904d0145a99c08d15743bd"),
    "kjd": (321207976, "3eefd5de96a73e60bac2e58db0bc1096"),
    "l38": (319555472, "82658918d9bfd74ae511f227a7224343"),
    "n31": (317742648, "226c271e17ecea4ddb7ed844afc4212f"),
    "ngh": (352395448, "e59df9850310836a42df52caa9012a2b"),
    "p5d": (346957672, "a9128090b639d6c0b97532a12bef7c47"),
    "p9p": (299989752, "b35315f9235f273e6b5363aa43e28a30"),
    "qm9": (315823416, "68ef8616be330628e521e38d75d2eaa4"),
    "ssx": (303828216, "fd7dcc41fc08eb4213e2985580705a75"),
    "trh": (345358264, "f82b2148513c5b9e5924146934e30bab"),
    "tz8": (348397048, "82cc5f445aade45c7d3d6aff8d143577"),
    "u7y": (317955896, "de7445d86d476185a62da918eed15c8e"),
    "w4p": (333469688, "b1a570b7c305275c050467702c9ab2db"),
}


def fetch_verified_range(url, start, end, etag):
    headers = {"Range": f"bytes={start}-{end}", "Accept-Encoding": "identity"}
    request = Request(url, headers=headers)
    for attempt in range(3):
        try:
            with urlopen(request, timeout=120) as response:
                served = response.headers.get("Content-Range", "").split("/")[0]
                if (
                    response.status != 206
                    or served != f"bytes {start}-{end}"
                    or response.headers.get("ETag", "").strip('"') != etag
                ):
                    raise ValueError(
                        "ETH file or HTTP range changed. Attach original WildPPG participant MATs."
                    )
                data = response.read(end - start + 2)
                if len(data) != end - start + 1:
                    raise OSError("Incomplete WildPPG transfer")
                return data
        except (URLError, OSError) as exc:
            if attempt == 2:
                raise RuntimeError(
                    "Check the runtime's Internet access or upload the original WildPPG MAT files."
                ) from exc
            time.sleep(2**attempt)


def ensure_wildppg_inputs(directory, ids=None):
    """Create or resume the chest recording for each requested participant in a cache."""
    root = Path(directory)
    root.mkdir(parents=True, exist_ok=True)
    selected = list(STERNUM_FILES) if ids is None else list(dict.fromkeys(ids))
    if not selected or set(selected) - STERNUM_FILES.keys():
        raise ValueError(f"Select one or more participant IDs from {list(STERNUM_FILES)}")
    paths = []
    for pid in selected:
        expected, etag = STERNUM_FILES[pid]
        name = f"WildPPG_Part_{pid}.mat"
        final = root / name
        part = root / f"{name}.{etag}.part"
        if not final.exists():
            offset = part.stat().st_size if part.exists() else 0
            if offset > expected or shutil.disk_usage(root).free < expected - offset:
                raise OSError(f"Check free space and partial file size for {name}")
            print(f"{pid}: fetching {(expected-offset)/1e6:.0f} MB of sternum data", flush=True)
            with part.open("ab") as output:
                while offset < expected:
                    end = min(offset + 16 * 1024**2, expected) - 1
                    output.write(fetch_verified_range(WILDPPG_BASE + name, offset, end, etag))
                    offset = end + 1
            if [variable for variable, _, _ in whosmat(part, appendmat=False)] != [
                "id",
                "notes",
                "sternum",
            ]:
                raise ValueError(
                    f"Unexpected MATLAB structure in {name}; attach the original recording."
                )
            part.replace(final)
        if final.stat().st_size != expected:
            raise ValueError(
                f"Unexpected file in download cache: {final}. Use a separate empty cache directory."
            )
        paths.append(final)
    return paths

### Find, verify and read recordings

In [ ]:
# Only WILDPPG_TRAIN_IDS are read here. WILDPPG_TEST_IDS are read once, after every model and
# threshold is frozen.
from joblib import Parallel, delayed

WILDPPG_FS = 128
# each worker holds one ~12-h recording in memory (~1 GB)
N_JOBS = int(os.getenv("PULSE_JOBS", "1"))

WILDPPG_MANIFEST = {}


def find_wildppg(ids):
    found = {}
    for root in [DATA_ROOT, WILDPPG_CACHE]:
        if root.is_dir():
            for p in root.rglob("WildPPG_Part_*.mat"):
                found.setdefault(p.stem.split("_")[-1], p)
    missing = [i for i in ids if i not in found]
    if missing and ALLOW_DOWNLOAD:
        for p in ensure_wildppg_inputs(WILDPPG_CACHE, missing):
            found[p.stem.split("_")[-1]] = p
    absent = [i for i in ids if i not in found]
    if absent:
        raise FileNotFoundError(f"WildPPG participants unavailable: {absent}")
    for pid in ids:
        path = found[pid]
        expected, etag = STERNUM_FILES[pid]
        if path.stat().st_size < expected:
            raise ValueError(
                f"{path}: cropped/unverified prefix; use the complete original or pinned sternum prefix"
            )
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1 << 22), b""):
                digest.update(block)
        WILDPPG_MANIFEST[pid] = {
            "bytes": path.stat().st_size,
            "sha256": digest.hexdigest(),
            "pinned_source_etag": etag,
            "provenance": "local file fingerprint; downloader verifies source ETag/range; structure and rates checked on read",
        }
    (OUT / "wildppg_source_manifest.json").write_text(
        json.dumps(WILDPPG_MANIFEST, indent=2), encoding="utf-8"
    )
    return {i: found[i] for i in ids}


def read_sternum(path):
    mat = sio.loadmat(path, variable_names=["id", "sternum"], simplify_cells=True)
    s = mat["sternum"]
    for k in ["ppg_ir", "acc_x", "acc_y", "acc_z", "ecg"]:
        if not np.isclose(float(s[k]["fs"]), WILDPPG_FS):
            raise ValueError(f"{path.name}: unexpected {k} sampling rate")
    pid = str(mat["id"]).strip()
    if pid != path.stem.split("_")[-1]:
        raise ValueError("Participant ID mismatch")
    lengths = [len(s[k]["v"]) for k in ["ppg_ir", "acc_x", "acc_y", "acc_z", "ecg"]]
    if len(set(lengths)) != 1:
        raise ValueError("Unsynchronized WildPPG arrays")
    n = None if WILDPPG_MAX_S is None else WILDPPG_MAX_S * WILDPPG_FS
    acc = np.vstack([np.asarray(s[k]["v"], float)[:n] for k in ["acc_x", "acc_y", "acc_z"]])
    # Raw optical intensity falls when blood volume rises; negate to the clinical PLETH convention
    # (as v4).
    out = (
        str(mat["id"]).strip(),
        -np.asarray(s["ppg_ir"]["v"], float)[:n],
        np.linalg.norm(acc, axis=0),
        np.asarray(s["ecg"]["v"], float)[:n],
    )
    del mat, s
    return out

### ECG reference beat count

Used only to label pulse visibility (PPG beats ÷ ECG beats). This Pan–Tompkins-inspired counter replaces v4's, which double-counted beats; see the folder README.

In [ ]:
PT_SOS = signal.butter(2, [5.0, 15.0], fs=WILDPPG_FS, btype="band", output="sos")


def ecg_beats(segment):
    # Reference beat count (Pan-Tompkins-inspired: 5-15 Hz band-pass, derivative, square, 150-ms
    # integration, 0.3-s refractory). v4's counter (0.25-s spacing, raw-amplitude peaks)
    # double-counted, probably T waves: against an independent ECG-autocorrelation HR (900 windows,
    # an0/e61/p5d) it had a median error of 37.5 bpm (median HR 124), while this one has 2.1 bpm
    # (77% within 10 bpm). Offline reference only.
    if len(segment) < WILDPPG_FS or not np.isfinite(segment).all() or np.ptp(segment) < 1e-10:
        return np.nan
    filtered = signal.sosfiltfilt(PT_SOS, segment)
    d = np.diff(filtered, prepend=filtered[0]) ** 2
    w = int(0.15 * WILDPPG_FS)
    env = np.convolve(d, np.ones(w) / w, mode="same")
    peaks, _ = signal.find_peaks(
        env, distance=int(0.3 * WILDPPG_FS), height=0.3 * np.percentile(env, 99)
    )
    return len(peaks)

### Per-window table: detector features, gate features, ECG beats and motion

In [ ]:
def skew_and_template(z, peaks):
    # Scale-free PPG-only quality features: skewness and mean correlation of each beat with the
    # median beat.
    sk = float(stats.skew(z))
    lo, hi = int(0.3 * DEVICE_FS), int(0.5 * DEVICE_FS)
    beats = np.array([z[p - lo : p + hi] for p in peaks if p - lo >= 0 and p + hi <= len(z)])
    if len(beats) < 3:
        return sk, 0.0
    template = np.median(beats, axis=0)
    return sk, float(np.nanmean([np.corrcoef(b, template)[0, 1] for b in beats]))


GATE_FEATURES = FEATURES + ["skewness", "template_r"]


def gate_row(w):
    F, peaks = ppg_features(w)
    sk, tr = skew_and_template(prepare(w), peaks)
    return {**F, "skewness": sk, "template_r": tr}


def window_table(pid, path, step_s, keep_waves):
    # One row per 16-s window: detector features, gate features, ECG reference beats and motion.
    _, ppg, acc, ecg = read_sternum(path)
    ppg, acc = to_device_rate(ppg, WILDPPG_FS), to_device_rate(acc, WILDPPG_FS)
    win, step = WINDOW_S * DEVICE_FS, step_s * DEVICE_FS
    starts = [s for s in range(0, len(ppg) - win + 1, step) if np.isfinite(ppg[s : s + win]).all()]
    rows, waves = [], []
    for s in starts:
        w = ppg[s : s + win]
        e0 = int(s / DEVICE_FS * WILDPPG_FS)
        rows.append(
            {
                "participant": pid,
                "start_s": s / DEVICE_FS,
                "motion_sd": float(np.nanstd(acc[s : s + win])),
                "ecg_beats": ecg_beats(ecg[e0 : e0 + WINDOW_S * WILDPPG_FS]),
                **gate_row(w),
            }
        )
        if keep_waves:
            waves.append(w.astype(np.float32))
    del ppg, acc, ecg
    frame = pd.DataFrame(rows)
    frame["visibility"] = frame.n_beats / frame.ecg_beats.where(frame.ecg_beats > 0)
    frame["visible"] = frame.visibility.between(*VISIBLE_RATIO)
    return frame, (np.vstack(waves) if keep_waves and waves else None)

Score every training participant with non-overlapping 16-s windows.

In [ ]:
started = time.monotonic()
train_parts = Parallel(n_jobs=N_JOBS)(
    delayed(window_table)(pid, path, TRAIN_STEP_S, True)
    for pid, path in find_wildppg(WILDPPG_TRAIN_IDS).items()
)
gate_train = pd.concat([f for f, _ in train_parts], ignore_index=True)
train_waves = np.vstack([w for _, w in train_parts])
del train_parts
labelled = gate_train.ecg_beats.gt(0).to_numpy()  # no ECG reference count -> no visibility label
print(
    f"WildPPG training participants: {len(WILDPPG_TRAIN_IDS)}, {len(gate_train)} windows "
    f"({len(gate_train) * TRAIN_STEP_S / 3600:.1f} h), pulse visible in {gate_train.visible[labelled].mean():.0%} "
    f"of labelled windows ({time.monotonic() - started:.0f} s)"
)

### Fit the pulse-visibility gate

PPG-only features predict whether the PPG beat count agrees with ECG. The threshold is the lowest whose leave-one-participant-out precision reaches `GATE_MIN_PRECISION`; tied scores are accepted or rejected together.

In [ ]:
# --- Pulse-visibility gate: PPG-only features -> P(PPG beats agree with ECG beats).
# Leave-one-participant-out.
def make_gate():
    return HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        min_samples_leaf=20,
        l2_regularization=1.0,
        random_state=SEED,
    )


def precision_threshold(scores, labels, target):
    # Lowest threshold whose precision (fraction truly visible among windows at/above it) is >=
    # target.
    order = np.argsort(-scores, kind="stable")
    s, l = scores[order], labels[order]
    precision = np.cumsum(l) / np.arange(1, len(l) + 1)
    # Evaluate whole tied-score groups: >= threshold accepts EVERY tied row.
    group_end = np.r_[s[:-1] != s[1:], True] if len(s) else np.zeros(0, bool)
    ok = np.flatnonzero((precision >= target) & group_end)
    return float(s[ok.max()]) if len(ok) else float("inf")


Xg, yg, groups = (
    gate_train[GATE_FEATURES].to_numpy(float),
    gate_train.visible.to_numpy(int),
    gate_train.participant.to_numpy(),
)
gate_lopo = np.full(len(gate_train), np.nan)
for pid in WILDPPG_TRAIN_IDS:
    te, tr = labelled & (groups == pid), labelled & (groups != pid)
    if te.any() and len(np.unique(yg[tr])) == 2:
        gate_lopo[te] = make_gate().fit(Xg[tr], yg[tr]).predict_proba(Xg[te])[:, 1]
ok = labelled & np.isfinite(gate_lopo)
GATE_THRESHOLD = precision_threshold(gate_lopo[ok], yg[ok], GATE_MIN_PRECISION)
if not ok.any() or len(np.unique(yg[labelled])) != 2:
    raise ValueError(
        "Insufficient reference-agreement classes for gate fitting; inspect ECG, never invent labels"
    )
gate_model = make_gate().fit(Xg[labelled], yg[labelled])
gate_rows = [
    {
        "split": "train participants (leave-one-out)",
        "windows": int(ok.sum()),
        "visible_fraction": yg[ok].mean(),
        "AUROC": roc_auc_score(yg[ok], gate_lopo[ok]),
        "precision": (
            yg[ok][gate_lopo[ok] >= GATE_THRESHOLD].mean()
            if (gate_lopo[ok] >= GATE_THRESHOLD).any()
            else np.nan
        ),
        "coverage": (gate_lopo[ok] >= GATE_THRESHOLD).mean(),
    }
]
print(
    "Gate threshold",
    round(GATE_THRESHOLD, 4),
    "|",
    {k: (round(v, 3) if isinstance(v, float) else v) for k, v in gate_rows[0].items()},
)

### Healthy negatives and the motion-artefact bank

`HEALTHY_NEG_PER_PARTICIPANT` pulse-visible windows per training participant become extra negatives for the `*_h` models. High-motion windows without a visible pulse are kept for augmentation.

In [ ]:
# --- Healthy negatives (pulse visible, non-overlapping) and a motion-artefact bank, training
# participants only.
rng_h = np.random.default_rng(SEED + 5)
pick = []
for pid, g in gate_train.groupby("participant"):
    pool = g.index[(g.visible & g.ecg_beats.gt(0)).to_numpy()]
    pick += list(rng_h.choice(pool, min(HEALTHY_NEG_PER_PARTICIPANT, len(pool)), replace=False))
pick = np.sort(np.array(pick, int))
H_wave = train_waves[pick]
H_feat = gate_train.loc[pick, FEATURES].to_numpy(float)
H_norm = np.vstack([prepare(w) for w in H_wave]).astype(np.float32)
moving = np.flatnonzero(
    (gate_train.motion_sd >= gate_train.motion_sd.quantile(0.9)).to_numpy()
    & ~gate_train.visible.to_numpy()
)
ARTEFACT_BANK = np.vstack(
    [prepare(train_waves[i]) for i in rng_h.choice(moving, min(300, len(moving)), replace=False)]
).astype(np.float32)
del train_waves
print(
    f"{len(H_norm)} healthy negatives; {len(ARTEFACT_BANK)} high-motion artefact windows for augmentation"
)

## Augmentation and the `cnn_v5` ensemble

Augmented copies keep the original expert label. They add respiratory amplitude modulation (≤ 30%), sometimes a 2–6 s motion-artefact burst from a healthy chest recording, and band-limited sensor noise. There is no time stretching, because that changes heart rate.

In [ ]:
# Features for the extra ICU windows, and the gate score on each record's prior window.
from sklearn.model_selection import train_test_split

feature_extra = {k: pd.DataFrame([ppg_features(w)[0] for w in X_extra[k]]) for k in X_extra}
X_feat_extra = {k: f[FEATURES].to_numpy(float) for k, f in feature_extra.items()}
X_norm_extra = {k: np.vstack([prepare(w) for w in X_extra[k]]).astype(np.float32) for k in X_extra}
icu_gate_pre = gate_model.predict_proba(
    pd.DataFrame([gate_row(w) for w in X_extra["pre"]])[GATE_FEATURES].to_numpy(float)
)[:, 1]

### Signal-corruption augmentation

In [ ]:
def augment(z, rng):
    # Retains the underlying expert event label; corruption can obscure the observed pulse (no time
    # stretching, which changes heart rate).
    n = len(z)
    t = np.arange(n) / DEVICE_FS
    out = z * (
        1
        + rng.uniform(0, 0.3)
        * np.sin(2 * np.pi * rng.uniform(0.15, 0.4) * t + rng.uniform(0, 2 * np.pi))
    )
    # motion artefact burst from a healthy chest recording (training participants)
    if rng.random() < 0.5:
        a, length = ARTEFACT_BANK[rng.integers(len(ARTEFACT_BANK))], int(
            rng.uniform(2, 6) * DEVICE_FS
        )
        s = rng.integers(0, n - length)
        out[s : s + length] += rng.uniform(0.2, 1.0) * a[s : s + length] * np.hanning(length)
    # band-limited sensor noise
    out = out + signal.sosfiltfilt(SOS, rng.normal(0, rng.uniform(0, 0.05), n))
    spread = np.percentile(out, 95) - np.percentile(out, 5)
    return (out / spread if spread > 0 else out).astype(np.float32)


rng_aug = np.random.default_rng(SEED + 7)
AUGMENTED_ICU = np.stack(
    [np.vstack([augment(z, rng_aug) for z in X_norm]) for _ in range(AUG_COPIES)]
).astype(np.float32)

### `cnn_v5`

The receptive field is ≈ 4.9 s, so a 4-s gap fits inside it; eight ≈ 2-s max-pooled bins keep *where* a beat occurred. Each of the three seeds stops early on its own stratified 15% of the ICU training records.

In [ ]:
if HAVE_TORCH:

    class PulseCNN5(nn.Module):
        # Receptive field ~4.9 s (a 4-s gap fits inside it); eight ~2-s max-pooled bins keep WHERE a
        # beat occurred.
        def __init__(self):
            super().__init__()
            self.body = nn.Sequential(
                nn.Conv1d(1, 8, 9, stride=2, padding=4),
                nn.ReLU(),
                nn.MaxPool1d(2),
                nn.Conv1d(8, 16, 7, padding=3),
                nn.ReLU(),
                nn.MaxPool1d(2),
                nn.Conv1d(16, 16, 7, padding=3),
                nn.ReLU(),
                nn.MaxPool1d(2),
                nn.Conv1d(16, 24, 5, padding=4, dilation=2),
                nn.ReLU(),
                nn.MaxPool1d(2),
                nn.Conv1d(24, 24, 5, padding=4, dilation=2),
                nn.ReLU(),
                nn.AdaptiveMaxPool1d(8),
            )
            self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.2), nn.Linear(24 * 8, 1))

        def forward(self, x):
            return self.head(self.body(x.unsqueeze(1))).squeeze(-1)

    def _proba(net, X):
        net.eval()
        with torch.no_grad():
            return np.concatenate(
                [
                    torch.sigmoid(net(torch.tensor(np.asarray(X[i : i + 256], np.float32)))).numpy()
                    for i in range(0, len(X), 256)
                ]
            )

    class CNN5Model:
        # 3-seed ensemble. Each member stops early on its own stratified 15% of the ICU training
        # records.
        def fit(self, idx, healthy, monitor=None):
            self.nets, self.history, self.best_epochs = [], [], []
            for k in range(CNN5_SEEDS):
                seed = SEED + 1000 + k
                tr, va = train_test_split(
                    idx, test_size=CNN5_VALID_FRACTION, stratify=y[idx], random_state=seed
                )
                Xtr, ytr = X_norm[tr], y[tr]
                if healthy:
                    Xtr, ytr = np.vstack([Xtr, H_norm]), np.r_[ytr, np.zeros(len(H_norm), int)]
                net, best = self._train(
                    seed,
                    Xtr,
                    ytr,
                    AUGMENTED_ICU[:, tr],
                    X_norm[va],
                    y[va],
                    (monitor or {}) if k == 0 else {},
                )
                self.nets.append(net)
                self.best_epochs.append(best)
            return self

        def _train(self, seed, Xtr, ytr, aug, Xva, yva, monitor):
            torch.manual_seed(seed)
            rng = np.random.default_rng(seed)
            net = PulseCNN5()
            opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-3)
            pos = max(1, int(ytr.sum()))
            loss_fn = nn.BCEWithLogitsLoss(
                pos_weight=torch.tensor((len(ytr) - pos) / pos, dtype=torch.float32)
            )
            yt = torch.tensor(ytr, dtype=torch.float32)
            best, best_epoch, state, wait = np.inf, 0, None, 0
            for epoch in range(1, CNN5_MAX_EPOCHS + 1):
                Xe = Xtr.copy()
                # augmented copies exist for the ICU rows (the first n_aug rows)
                n_aug = aug.shape[1]
                swap = np.flatnonzero(rng.random(n_aug) < AUG_PROB)
                Xe[swap] = aug[rng.integers(0, aug.shape[0], len(swap)), swap]
                Xe = torch.tensor(Xe)
                net.train()
                order = rng.permutation(len(ytr))
                for b in range(0, len(order), 32):
                    i = order[b : b + 32]
                    opt.zero_grad()
                    loss_fn(net(Xe[i]), yt[i]).backward()
                    opt.step()
                pv = _proba(net, Xva)
                vloss = balanced_log_loss(yva, pv)
                self.history.append({"step": epoch, "split": "early_stop", **split_scores(yva, pv)})
                for split, (Xm, ym) in monitor.items():
                    self.history.append(
                        {"step": epoch, "split": split, **split_scores(ym, _proba(net, Xm))}
                    )
                if vloss < best - 1e-4:
                    best, best_epoch, wait = vloss, epoch, 0
                    state = {k: v.detach().clone() for k, v in net.state_dict().items()}
                else:
                    wait += 1
                    if wait >= CNN5_PATIENCE:
                        break
            net.load_state_dict(state)
            net.eval()
            return net, best_epoch

        def predict_proba(self, X):
            p = np.mean([_proba(net, X) for net in self.nets], axis=0)
            return np.c_[1 - p, p]

    print("cnn_v5 parameters:", sum(p.numel() for p in PulseCNN5().parameters()))

### Fit and predict any model

`*_h` models also train on the healthy negatives, which are never used for validation. The v4 models are fitted exactly as in v4.

In [ ]:
V4_MODELS = ["rule_definitions", "logistic", "boosted_trees"] + (["cnn_1d"] if HAVE_TORCH else [])
MODELS = V4_MODELS + ["boosted_trees_h"] + (["cnn_v5", "cnn_v5_h"] if HAVE_TORCH else [])


def _fit(name, idx, monitor=None):
    """idx = ICU record indices. *_h models also train on the healthy negatives (never used for validation).
    v4 models are fitted exactly as in v4."""
    monitor = monitor or {}
    if name == "cnn_1d":
        return CNNModel().fit(
            X_norm[idx], y[idx], {s: (X_norm[i], y[i]) for s, i in monitor.items()}
        )
    if name in ("cnn_v5", "cnn_v5_h"):
        return CNN5Model().fit(
            idx, name.endswith("_h"), {s: (X_norm[i], y[i]) for s, i in monitor.items()}
        )
    base = name[:-2] if name.endswith("_h") else name
    Xf, yf = X_feat[idx], y[idx]
    if name.endswith("_h"):
        Xf, yf = np.vstack([Xf, H_feat]), np.r_[yf, np.zeros(len(H_feat), int)]
    model = make_feature_model(base).fit(Xf, yf)
    model.history = []
    if base == "boosted_trees":
        for split, i in monitor.items():
            for step, p in enumerate(model.staged_predict_proba(X_feat[i]), 1):
                model.history.append({"step": step, "split": split, **split_scores(y[i], p[:, 1])})
    return model


def predict(name, model, idx, which=None):
    """which=None: the window ending at the alarm; 'm2'/'m4'/'pre': the extra ICU windows."""
    F = feature_table if which is None else feature_extra[which]
    if name == "rule_definitions":
        return rule_definitions(F.iloc[idx])
    if name.startswith("cnn"):
        return model.predict_proba((X_norm if which is None else X_norm_extra[which])[idx])[:, 1]
    return model.predict_proba((X_feat if which is None else X_feat_extra[which])[idx])[:, 1]


print("Architectures:", MODELS)

### Checkpoint cache

Each fit is cached under a hash of the version key, run plan, model, record indices, healthy-source fingerprints and library versions.

In [ ]:
def fit(name, idx, monitor=None):
    key_data = {
        "code": AUDITED_CODE_SHA256,
        "plan": PREREG,
        "model": name,
        "indices": list(map(int, idx)),
        "healthy_training_sources": {k: WILDPPG_MANIFEST[k] for k in WILDPPG_TRAIN_IDS},
        "monitor": {k: list(map(int, v)) for k, v in (monitor or {}).items()},
        "sklearn": sklearn.__version__,
        "torch": torch.__version__ if HAVE_TORCH else None,
    }
    key = hashlib.sha256(json.dumps(key_data, sort_keys=True).encode()).hexdigest()
    dest = CACHE / "v5_trusted_local_fits" / (key + ".joblib")
    if dest.exists():
        # Only this notebook's trusted local cache; never load external joblib files.
        return joblib.load(dest)
    result = _fit(name, idx, monitor)
    dest.parent.mkdir(parents=True, exist_ok=True)
    temp = dest.with_suffix(".tmp")
    joblib.dump(result, temp)
    temp.replace(dest)
    return result

## Cross-validation on ICU records

- **Outer 5-fold cross-validation**, stratified by alarm type × label. Each record is scored once by models that never saw it. Patient IDs are not published, so folds are record-level.
- **Thresholds come only from training data.** Inside each outer training fold, a 5-fold inner cross-validation produces training-only scores. Two operating points are then chosen:
  - `high_sensitivity`: the largest threshold with inner sensitivity ≥ 0.90;
  - `high_specificity`: the smallest threshold with inner specificity ≥ 0.90.

  `rule_definitions` has no threshold.
- **Reported:**
  - AUROC with a bootstrap 95% CI;
  - sensitivity and specificity at each operating point, with Wilson 95% CIs;
  - per-alarm-type results;
  - accuracy and balanced accuracy (the mean of sensitivity and specificity). 76% of records are false alarms, so a detector that never fires already has 0.76 accuracy; read balanced accuracy alongside it;
  - the Challenge score, (TP + TN) / (TP + TN + FP + 5·FN), which counts a missed event five times as heavily as a false alarm.
- **Training diagnostics** (next section) are recorded from the same fitted models, so they do not change any result.

In [ ]:
def thresholds_from(scores, labels):
    pos, neg = np.sort(scores[labels == 1]), np.sort(scores[labels == 0])
    if not len(pos) or not len(neg):
        return {"high_sensitivity": np.nan, "high_specificity": np.nan}
    hs = pos[int(np.floor((1 - TARGET_SENSITIVITY) * len(pos)))]  # >= 90% of positives at or above
    hp = neg[min(len(neg) - 1, int(np.ceil(TARGET_SPECIFICITY * len(neg))) - 1)]
    return {"high_sensitivity": float(hs), "high_specificity": float(np.nextafter(hp, np.inf))}


strata = (meta.alarm + "_" + meta.label.astype(str)).to_numpy()
outer = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=SEED)
oof = {m: np.full(len(y), np.nan) for m in MODELS}
oof_extra = {m: {k: np.full(len(y), np.nan) for k in X_extra} for m in MODELS}  # extra ICU windows
fold_of = np.full(len(y), -1)
chosen = []
# Diagnostics: train = outer-training records (in-sample), valid = inner held-out folds (the split
# that chooses thresholds), test = outer held-out fold (never seen in fitting or threshold choice).
curves, split_rows, roc_parts = [], [], []
started = time.monotonic()
for fold, (train, test) in enumerate(outer.split(X_feat, strata)):
    fold_of[test] = fold
    inner = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=SEED + fold)
    for m in MODELS:
        if m == "rule_definitions":
            oof[m][test] = predict(m, None, test)
            for k in X_extra:
                oof_extra[m][k][test] = predict(m, None, test, k)
            split_rows.append(
                {
                    "model": m,
                    "fold": fold,
                    "split": "test",
                    "records": len(test),
                    "AUROC": np.nan,
                    # 0/1 output: no loss
                    **split_scores(y[test], oof[m][test]),
                    "loss": np.nan,
                }
            )
            continue
        print(f"fold {fold+1}/{OUTER_FOLDS}: {m}", flush=True)
        inner_scores = np.full(len(train), np.nan)
        for k, (itr, ite) in enumerate(inner.split(X_feat[train], strata[train])):
            model = fit(m, train[itr], monitor={"valid": train[ite]})
            inner_scores[ite] = predict(m, model, train[ite])
            curves += [{"model": m, "fold": fold, "inner": k, **h} for h in model.history]
        th = thresholds_from(inner_scores, y[train])
        chosen.append({"fold": fold, "model": m, **th})
        model = fit(m, train, monitor={"train": train, "test": test})
        curves += [{"model": m, "fold": fold, "inner": -1, **h} for h in model.history]
        oof[m][test] = predict(m, model, test)
        for k in X_extra:
            oof_extra[m][k][test] = predict(m, model, test, k)
        for split, idx, s in [
            ("train", train, predict(m, model, train)),
            ("valid", train, inner_scores),
            ("test", test, oof[m][test]),
        ]:
            split_rows.append(
                {
                    "model": m,
                    "fold": fold,
                    "split": split,
                    "records": len(idx),
                    "AUROC": roc_auc_score(y[idx], s),
                    **split_scores(y[idx], s),
                }
            )
            roc_parts.append(
                {"model": m, "split": split, "fold": fold, "labels": y[idx], "scores": s}
            )
    print(
        f"outer fold {fold + 1}/{OUTER_FOLDS} done ({time.monotonic() - started:.0f} s)", flush=True
    )
chosen = pd.DataFrame(chosen)
curves, split_metrics = pd.DataFrame(curves), pd.DataFrame(split_rows)

### Results at each operating point

AUROC with a bootstrap 95% CI, sensitivity and specificity with Wilson 95% CIs, and per-alarm-type counts.

In [ ]:
def bootstrap_auc(scores, labels, n=1000):
    rng = np.random.default_rng(SEED)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(labels), len(labels))
        if len(np.unique(labels[i])) == 2:
            vals.append(roc_auc_score(labels[i], scores[i]))
    return np.percentile(vals, [2.5, 97.5])


def flags_for(m, point):
    if m == "rule_definitions":
        return oof[m] >= 0.5
    th = chosen.set_index(["fold", "model"])[point]
    return np.array([oof[m][i] >= th[(fold_of[i], m)] for i in range(len(y))])


rows, per_type = [], []
for m in MODELS:
    auc = roc_auc_score(y, oof[m]) if m != "rule_definitions" else np.nan
    auc_lo, auc_hi = bootstrap_auc(oof[m], y) if m != "rule_definitions" else (np.nan, np.nan)
    for point in (
        ["definition"] if m == "rule_definitions" else ["high_sensitivity", "high_specificity"]
    ):
        flag = flags_for(m, "high_sensitivity" if point == "definition" else point)
        tp, fn = int((flag & (y == 1)).sum()), int((~flag & (y == 1)).sum())
        tn, fp = int((~flag & (y == 0)).sum()), int((flag & (y == 0)).sum())
        se, sp = tp / (tp + fn), tn / (tn + fp)
        rows.append(
            {
                "model": m,
                "operating_point": point,
                "AUROC": auc,
                "AUROC_lo": auc_lo,
                "AUROC_hi": auc_hi,
                "sensitivity": se,
                "sens_lo": proportion_ci(tp, tp + fn)[0],
                "sens_hi": proportion_ci(tp, tp + fn)[1],
                "specificity": sp,
                "spec_lo": proportion_ci(tn, tn + fp)[0],
                "spec_hi": proportion_ci(tn, tn + fp)[1],
                "accuracy": (tp + tn) / len(y),
                "balanced_accuracy": (se + sp) / 2,
                "challenge_score": (tp + tn) / (tp + tn + fp + 5 * fn),
                "TP": tp,
                "FN": fn,
                "TN": tn,
                "FP": fp,
            }
        )
        for alarm in TARGET_ALARMS:
            a = (meta.alarm == alarm).to_numpy()
            per_type.append(
                {
                    "model": m,
                    "operating_point": point,
                    "alarm": alarm,
                    "true_events": int(y[a].sum()),
                    "caught": int((flag & a & (y == 1)).sum()),
                    "false_alarms": int((a & (y == 0)).sum()),
                    "wrongly_flagged": int((flag & a & (y == 0)).sum()),
                }
            )
results, per_type = pd.DataFrame(rows), pd.DataFrame(per_type)
display(results.round(3))
display(
    per_type.pivot_table(
        index=["model", "operating_point"],
        columns="alarm",
        values=["caught", "wrongly_flagged"],
        aggfunc="sum",
    )
)

## Persistence and the prior-window gate on ICU records

In [ ]:
# Secondary ICU analyses (same out-of-fold models and fold thresholds as the table above).
#  * 2-of-3 persistence: windows ending 4, 2 and 0 s before the alarm; flag if >= PERSIST_K of the 3
#    flag.
#  * Prior-window gate: would the gate (trained on healthy chest PPG) have been open on 268-284 s?
#    Finger ICU PPG differs from chest PPG, so this is only an indication of how much sensitivity
#    the gate could cost.
def flags_on(m, point, scores):
    if m == "rule_definitions":
        return scores >= 0.5
    th = chosen.set_index(["fold", "model"])[point]
    return np.array([scores[i] >= th[(fold_of[i], m)] for i in range(len(y))])


persist_rows = []
for m in MODELS:
    for point in (
        ["definition"] if m == "rule_definitions" else ["high_sensitivity", "high_specificity"]
    ):
        key = "high_sensitivity" if point == "definition" else point
        f0 = flags_on(m, key, oof[m])
        votes = (
            f0.astype(int)
            + flags_on(m, key, oof_extra[m]["m2"])
            + flags_on(m, key, oof_extra[m]["m4"])
        )
        for config, flag in [
            ("single window", f0),
            (f"{PERSIST_K} of {PERSIST_N}", votes >= PERSIST_K),
            ("prior-window gate", f0 & (icu_gate_pre >= GATE_THRESHOLD)),
            ("gate + persistence", (votes >= PERSIST_K) & (icu_gate_pre >= GATE_THRESHOLD)),
        ]:
            tp, tn = int((flag & (y == 1)).sum()), int((~flag & (y == 0)).sum())
            persist_rows.append(
                {
                    "model": m,
                    "operating_point": point,
                    "configuration": config,
                    "sensitivity": tp / y.sum(),
                    "sens_lo": proportion_ci(tp, y.sum())[0],
                    "sens_hi": proportion_ci(tp, y.sum())[1],
                    "specificity": tn / (y == 0).sum(),
                }
            )
persistence_icu = pd.DataFrame(persist_rows)
display(persistence_icu.round(3))
gate_open_pre = icu_gate_pre >= GATE_THRESHOLD
icu_gate_pass = (
    meta.assign(gate_open=gate_open_pre)[y == 1]
    .groupby("alarm")
    .gate_open.agg(true_events="size", gate_open="sum")
    .assign(fraction=lambda d: d.gate_open / d.true_events)
)
print("Prior-window gate open before expert-confirmed events (ICU finger PPG, indicative only):")
display(icu_gate_pass)

## Training diagnostics: loss, accuracy and ROC on train, validation and test

Every number here comes from the models fitted in the cross-validation above. There are three splits per outer fold:

| Split | Records | Scored by |
|---|---|---|
| **train** | the outer fold's training records (about 411) | the outer-fold model, in-sample |
| **valid** | the same training records, each scored by an inner-CV model that did not see it | the 5 inner-fold models. This is the split that chooses the thresholds |
| **test** | the outer held-out fold (about 103) | the outer-fold model, which never saw these records |

- **Loss** is the class-balanced log loss: the mean of the true-event and false-alarm mean losses. It is proportional to what every learned model minimises during training, so the three splits are directly comparable.
- **Accuracy** and **balanced accuracy** here use a 0.5 cut-off, the natural decision point of a class-balanced model. The operating-point results above use the chosen thresholds instead.
- **Curves** are exported for the original iterative baselines: `boosted_trees` (per boosting round) and `cnn_1d` (per epoch). Logistic regression has only final values. `rule_definitions` has nothing trained; only its test row is shown. The v5 CNN uses a separate internal training split for early stopping.
- **The outer-test curves are diagnostic only.** The original baselines retain 40 epochs and 200 boosting rounds. The v5 CNN stops using its internal training validation loss; outer-test results do not select its epoch.
- **How to read the gap:** a train loss far below the valid and test losses means overfitting. A test value close to valid suggests the thresholds chosen on valid will transfer.

In [ ]:
summary = split_metrics.groupby(["model", "split"], sort=False)[
    ["AUROC", "loss", "accuracy", "balanced_accuracy"]
].agg(["mean", "min", "max"])
print("Mean over the 5 outer folds (min-max across folds):")
display(summary.round(3))
gap = split_metrics.pivot_table(
    index="model", columns="split", values=["AUROC", "loss"], aggfunc="mean"
)
gap = gap.dropna(axis=0, how="any")
if len(gap):
    display(
        pd.DataFrame(
            {
                "AUROC train - test": gap[("AUROC", "train")] - gap[("AUROC", "test")],
                "loss test - train": gap[("loss", "test")] - gap[("loss", "train")],
            }
        ).round(3)
    )
if len(curves):
    last = curves[curves.step.eq(curves.groupby("model").step.transform("max"))]
    print("Final-step values of the training curves (mean over curves):")
    display(
        last.groupby(["model", "split"])[["loss", "accuracy", "balanced_accuracy"]].mean().round(3)
    )

## False alarms on healthy chest PPG: eight comparison participants

The split is person-disjoint within this run. These recordings are used for the inherited model-selection rule and are not an untouched validation set. Alarm-start rates must be read alongside time flagged, monitoring coverage, and gated ICU sensitivity.

First, every learned model is refitted on all ICU records, with thresholds from cross-validation on all records.

In [ ]:
final_models, final_thresholds = {}, {}
all_idx = np.arange(len(y))
cv_all = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=SEED + 99)
for m in MODELS:
    if m == "rule_definitions":
        continue
    scores = np.full(len(y), np.nan)
    for tr, te in cv_all.split(X_feat, strata):
        scores[te] = predict(m, fit(m, tr), te)
    final_thresholds[m] = thresholds_from(scores, y)
    final_models[m] = fit(m, all_idx)
print(
    "Final thresholds:",
    {m: {k: round(v, 4) for k, v in t.items()} for m, t in final_thresholds.items()},
)

### Score the comparison participants

Models, thresholds and the gate are frozen from here; only now are the comparison participants read. Every fifth window is also scored with the IR sign flipped, as a polarity check.

In [ ]:
def score_frame(F, Z):
    out = {"rule_definitions": rule_definitions(F)}
    for m, model in final_models.items():
        out[m] = model.predict_proba(Z if m.startswith("cnn") else F[FEATURES].to_numpy(float))[
            :, 1
        ]
    return out


FLIP_EVERY = 5


def score_participant(pid, path):
    if HAVE_TORCH:
        torch.set_num_threads(1 if N_JOBS > 1 else min(4, os.cpu_count() or 1))
    frame, waves = window_table(pid, path, TRANSFER_STEP_S, True)
    Z = np.vstack([prepare(w) for w in waves]).astype(np.float32)
    for m, s in score_frame(frame, Z).items():
        frame[f"score_{m}"] = s
    frame["gate_score"] = gate_model.predict_proba(frame[GATE_FEATURES].to_numpy(float))[:, 1]
    # Polarity check on every 5th window: the same frozen detectors on the opposite IR sign.
    sub = np.arange(0, len(waves), FLIP_EVERY)
    Fi = pd.DataFrame([ppg_features(-w)[0] for w in waves[sub]])
    Zi = np.vstack([prepare(-w) for w in waves[sub]]).astype(np.float32)
    flip = pd.DataFrame({f"score_{m}": s for m, s in score_frame(Fi, Zi).items()})
    flip["n_beats"], flip["participant"] = Fi.n_beats.to_numpy(), pid
    flip["ecg_beats"] = frame.ecg_beats.to_numpy()[sub]
    same = frame.iloc[sub][
        [c for c in frame.columns if c.startswith("score_")] + ["n_beats"]
    ].reset_index(drop=True)
    del waves, Z
    return frame, flip, same


started = time.monotonic()
parts = Parallel(n_jobs=N_JOBS)(
    delayed(score_participant)(pid, path) for pid, path in find_wildppg(WILDPPG_TEST_IDS).items()
)
transfer = pd.concat([p[0] for p in parts], ignore_index=True)
flip_opposite = pd.concat([p[1] for p in parts], ignore_index=True)
flip_same = pd.concat([p[2] for p in parts], ignore_index=True)
del parts
print(
    f"Test participants: {transfer.participant.nunique()}, {len(transfer) * TRANSFER_STEP_S / 3600:.1f} h "
    f"({time.monotonic() - started:.0f} s)"
)
transfer["motion"] = pd.qcut(
    transfer.motion_sd, 3, labels=["low motion", "mid motion", "high motion"]
)
transfer["pulse_visible"] = np.where(
    transfer.visible, "PPG pulse visible (0.8-1.2 x ECG beats)", "PPG pulse not visible"
)
# Prior-window gate: decided on the window that ends where the scored window starts (same
# participant).
prev = (
    transfer[["participant", "start_s", "gate_score"]]
    .assign(start_s=lambda d: d.start_s + WINDOW_S)
    .rename(columns={"gate_score": "gate_prev"})
)
transfer = transfer.merge(prev, on=["participant", "start_s"], how="left")
transfer["gate_open"] = transfer.gate_prev.ge(GATE_THRESHOLD).to_numpy()
print(
    "Median PPG/ECG beat ratio:",
    round(float(transfer.visibility.median()), 2),
    f"| visible pulse: {transfer.visible.mean():.0%} | prior-window gate open: {transfer.gate_open.mean():.0%}",
)

### Gate performance

In [ ]:
lab = transfer.ecg_beats.gt(0).to_numpy()
g_open_now = transfer.gate_score.to_numpy() >= GATE_THRESHOLD
gate_rows.append(
    {
        "split": "test participants",
        "windows": int(lab.sum()),
        "visible_fraction": transfer.visible[lab].mean(),
        "AUROC": roc_auc_score(transfer.visible[lab], transfer.gate_score[lab]),
        "precision": transfer.visible[lab & g_open_now].mean(),
        "coverage": g_open_now[lab].mean(),
    }
)
gate_performance = pd.DataFrame(gate_rows)
display(gate_performance.round(3))

### False-alert starts per monitored hour

An alert *starts* when a window is flagged and the previous window of the same participant (2 s earlier) was not.

In [ ]:
def consecutive(frame, k):
    return (
        frame.participant.eq(frame.participant.shift(k))
        & frame.start_s.sub(frame.start_s.shift(k)).eq(k * TRANSFER_STEP_S)
    ).to_numpy()


def start_mask(frame, flag):
    # An alert starts when a window is flagged and the previous window (same participant, 2 s
    # earlier) was not.
    return flag & ~(np.r_[False, flag[:-1]] & consecutive(frame, 1))


def persistent(frame, flag):
    votes = (
        flag.astype(int)
        + (np.r_[False, flag[:-1]] & consecutive(frame, 1))
        + (np.r_[False, False, flag[:-2]] & consecutive(frame, 2))
    )
    return votes >= PERSIST_K


fa_rows = []
gate = transfer.gate_open.to_numpy()
for m in MODELS:
    for point in (
        ["definition"] if m == "rule_definitions" else ["high_sensitivity", "high_specificity"]
    ):
        th = 0.5 if m == "rule_definitions" else final_thresholds[m][point]
        flag = (transfer[f"score_{m}"] >= th).to_numpy()
        configs = [
            ("single window", flag, np.ones(len(flag), bool)),
            ("prior-window gate", flag & gate, gate),
            (f"{PERSIST_K} of {PERSIST_N}", persistent(transfer, flag), np.ones(len(flag), bool)),
            (f"gate + {PERSIST_K} of {PERSIST_N}", persistent(transfer, flag) & gate, gate),
        ]
        for config, f, monitored in configs:
            starts = start_mask(transfer, f)
            groups = [("all", monitored)]
            if config == "single window":
                groups += [
                    (str(g), transfer.motion.eq(g).to_numpy())
                    for g in transfer.motion.cat.categories
                ]
                groups += [
                    (str(g), transfer.pulse_visible.eq(g).to_numpy())
                    for g in sorted(transfer.pulse_visible.unique())
                ]
            for group, mask in groups:
                hours = mask.sum() * TRANSFER_STEP_S / 3600
                n = int(starts[mask].sum())
                lo, hi = poisson_rate_ci(n, hours) if hours > 0 else (np.nan, np.nan)
                fa_rows.append(
                    {
                        "model": m,
                        "operating_point": point,
                        "configuration": config,
                        "subset": group,
                        "participants": transfer.participant[mask].nunique(),
                        "hours": hours,
                        "monitored_fraction": monitored.mean(),
                        "false_alert_starts": n,
                        "per_hour": n / hours if hours > 0 else np.nan,
                        "ci95_lo": lo,
                        "ci95_hi": hi,
                        "flagged_fraction": f[mask].mean() if mask.any() else np.nan,
                    }
                )
false_alarms = pd.DataFrame(fa_rows)
display(
    false_alarms[false_alarms.subset.eq("all")]
    .pivot_table(index=["model", "operating_point"], columns="configuration", values="per_hour")
    .round(1)
)

### Polarity check

In [ ]:
# Polarity: flagged fraction and PPG/ECG beat ratio with the v4 sign vs the opposite sign (every 5th
# window).
polarity_rows = []
for m in MODELS:
    point = "definition" if m == "rule_definitions" else "high_specificity"
    th = 0.5 if m == "rule_definitions" else final_thresholds[m][point]
    polarity_rows.append(
        {
            "model": m,
            "operating_point": point,
            "flagged_v4_sign": (flip_same[f"score_{m}"] >= th).mean(),
            "flagged_opposite_sign": (flip_opposite[f"score_{m}"] >= th).mean(),
        }
    )
ratio = lambda f: float((f.n_beats / f.ecg_beats.where(f.ecg_beats > 0)).median())
polarity_check = pd.DataFrame(polarity_rows).assign(
    beat_ratio_v4_sign=ratio(flip_same.assign(ecg_beats=flip_opposite.ecg_beats)),
    beat_ratio_opposite_sign=ratio(flip_opposite),
)
display(polarity_check.round(3))

### Apply the pre-declared selection rule

A provisional research ranking, not deployment authorization: see the limits in the introduction and in *Reading the results*.

In [ ]:
# Pre-declared firmware selection (SELECTION_RULE in the first cell).
rule_sens = results.loc[results.model.eq("rule_definitions"), "sensitivity"].iloc[0]
candidates = results[results.operating_point.isin(["definition", "high_specificity"])].merge(
    false_alarms[
        false_alarms.configuration.eq("prior-window gate") & false_alarms.subset.eq("all")
    ][["model", "operating_point", "per_hour", "ci95_hi", "monitored_fraction"]],
    on=["model", "operating_point"],
)
candidates["eligible"] = candidates.sensitivity >= rule_sens - 1e-12
candidates = candidates.sort_values(
    ["eligible", "per_hour", "AUROC"], ascending=[False, True, False], na_position="last"
)
selection = {
    "rule": SELECTION_RULE,
    "rule_definitions_sensitivity": float(rule_sens),
    "selected": (
        candidates[candidates.eligible & candidates.per_hour.notna()]
        .iloc[0][["model", "operating_point"]]
        .to_dict()
        if (candidates.eligible & candidates.per_hour.notna()).any() and not QUICK
        else None
    ),
    "status": "provisional research ranking only; not deployment authorization",
    "limitations": [
        "selection uses inspected WildPPG comparison",
        "original rule filters ungated sensitivity but ranks gated false alarms",
        "gated sensitivity and coverage must be reviewed together; see persistence_icu.csv",
    ],
    "candidates": candidates[
        [
            "model",
            "operating_point",
            "AUROC",
            "sensitivity",
            "specificity",
            "per_hour",
            "ci95_hi",
            "monitored_fraction",
            "eligible",
        ]
    ].to_dict("records"),
}
display(
    candidates[
        [
            "model",
            "operating_point",
            "AUROC",
            "sensitivity",
            "specificity",
            "per_hour",
            "ci95_hi",
            "monitored_fraction",
            "eligible",
        ]
    ].round(3)
)
print("Pre-declared selection:", selection["selected"])

## Figures

### 1 · Example windows: one true event and one false alarm per alarm type

In [ ]:
def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)


# 1. Example windows: one expert-confirmed true event and one false alarm per alarm type.
fig, axes = plt.subplots(len(TARGET_ALARMS), 2, figsize=(14, 2.2 * len(TARGET_ALARMS)), sharex=True)
t = np.arange(X_wave.shape[1]) / DEVICE_FS
for row, alarm in enumerate(TARGET_ALARMS):
    for col, label in enumerate([1, 0]):
        pick = np.flatnonzero((meta.alarm == alarm).to_numpy() & (y == label))
        ax = axes[row, col]
        if len(pick):
            z = prepare(X_wave[pick[0]])
            _, peaks = ppg_features(X_wave[pick[0]])
            ax.plot(t, z, lw=0.7)
            ax.plot(t[peaks], z[peaks], "r.", ms=4)
        ax.set_title(
            f"{alarm}: {'true event' if label else 'false alarm'} ({meta.record.iloc[pick[0]] if len(pick) else '-'})",
            fontsize=8,
        )
axes[-1, 0].set_xlabel("seconds before alarm (window end = alarm)")
axes[-1, 1].set_xlabel("seconds")
save(fig, "01_example_windows")

### 2 · Sensitivity vs specificity (outer cross-validation)

In [ ]:
# 2. Sensitivity vs specificity with 95% CIs (outer cross-validation).
fig, ax = plt.subplots(figsize=(7, 5))
for r in results.itertuples():
    ax.errorbar(
        1 - r.specificity,
        r.sensitivity,
        xerr=[[r.spec_hi - r.specificity], [r.specificity - r.spec_lo]],
        yerr=[[r.sensitivity - r.sens_lo], [r.sens_hi - r.sensitivity]],
        fmt="o",
        capsize=2,
    )
    ax.annotate(f"{r.model} ({r.operating_point})", (1 - r.specificity, r.sensitivity), fontsize=7)
ax.set(
    xlabel="1 - specificity (false monitor alarms flagged)",
    ylabel="sensitivity (expert-confirmed events caught)",
    xlim=(-0.02, 1.02),
    ylim=(-0.02, 1.02),
    title="Outer cross-validation, ICU finger PPG",
)
save(fig, "02_sensitivity_specificity")

### 3 · False alarms on healthy chest PPG

In [ ]:
# 3. False alarms on healthy chest PPG.
block = false_alarms[false_alarms.subset.eq("all") & false_alarms.configuration.eq("single window")]
fig, ax = plt.subplots(figsize=(9, 4))
labels = block.model + " / " + block.operating_point
ax.barh(
    labels,
    block.per_hour,
    xerr=[block.per_hour - block.ci95_lo, block.ci95_hi - block.per_hour],
    capsize=2,
)
ax.set(
    xlabel="false alert starts per hour (95% CI)", title="Healthy volunteers, sternum PPG (WildPPG)"
)
save(fig, "03_healthy_false_alarms")

### 4 · Training curves

In [ ]:
SPLIT_COLOURS = {"train": "C0", "valid": "C1", "test": "C3"}

# 4. Training curves: class-balanced log loss and accuracy (0.5 cut-off) per boosting round / CNN
#   epoch.
#    Line = mean over curves; band = min-max. valid = 25 inner fits; train and test = 5 outer fits.
iterative = [
    m
    for m in ["boosted_trees", "cnn_1d", "boosted_trees_h", "cnn_v5", "cnn_v5_h"]
    if len(curves) and m in set(curves.model)
]
if iterative:
    fig, axes = plt.subplots(len(iterative), 3, figsize=(15, 3.6 * len(iterative)), squeeze=False)
    for r, m in enumerate(iterative):
        c = curves[curves.model.eq(m)]
        for col, metric in enumerate(["loss", "accuracy", "balanced_accuracy"]):
            ax = axes[r, col]
            for split, colour in SPLIT_COLOURS.items():
                if not c.split.eq(split).any():
                    continue
                g = c[c.split.eq(split)].groupby("step")[metric]
                mean = g.mean()
                ax.plot(mean.index, mean, color=colour, label=split)
                ax.fill_between(mean.index, g.min(), g.max(), color=colour, alpha=0.15, lw=0)
            ax.set(
                title=f"{m}: {metric.replace('_', ' ')}",
                xlabel="boosting round" if m == "boosted_trees" else "epoch",
                ylabel=(
                    "class-balanced log loss"
                    if metric == "loss"
                    else f"{metric.replace('_', ' ')} at 0.5"
                ),
            )
        axes[r, 0].legend(fontsize=8)
    save(fig, "04_training_curves")

### 5 · ROC curves

In [ ]:
# 5. ROC curves. Left: pooled out-of-fold test scores (the AUROC in the results table) with the
#   rule's fixed point.
#    Others: per-fold ROC on train / valid / test, mean over folds, band = min-max.
learned = [m for m in MODELS if m != "rule_definitions"]
fig, axes = plt.subplots(
    1, len(learned) + 1, figsize=(4.6 * (len(learned) + 1), 4.4), squeeze=False
)
axes = axes[0]
for m in learned:
    fpr, tpr, _ = roc_curve(y, oof[m])
    axes[0].plot(fpr, tpr, label=f"{m} (AUROC {roc_auc_score(y, oof[m]):.2f})")
rule = results[results.model.eq("rule_definitions")].iloc[0]
axes[0].plot(1 - rule.specificity, rule.sensitivity, "k*", ms=11, label="rule_definitions (fixed)")
axes[0].set_title("Test: outer folds pooled")
grid = np.linspace(0, 1, 201)
for ax, m in zip(axes[1:], learned):
    for split, colour in SPLIT_COLOURS.items():
        tprs = []
        for part in roc_parts:
            if part["model"] == m and part["split"] == split:
                fpr, tpr, _ = roc_curve(part["labels"], part["scores"])
                tprs.append(np.interp(grid, fpr, tpr))
        tprs = np.array(tprs)
        auc = split_metrics[split_metrics.model.eq(m) & split_metrics.split.eq(split)].AUROC.mean()
        ax.plot(grid, tprs.mean(0), color=colour, label=f"{split} (mean AUROC {auc:.2f})")
        ax.fill_between(grid, tprs.min(0), tprs.max(0), color=colour, alpha=0.12, lw=0)
    ax.set_title(f"{m}: train / valid / test")
for ax in axes:
    ax.plot([0, 1], [0, 1], ":", color="grey", lw=0.8)
    ax.set(xlabel="1 - specificity", ylabel="sensitivity", xlim=(0, 1), ylim=(0, 1.01))
    ax.legend(fontsize=7, loc="lower right")
save(fig, "05_roc_curves")

### 6 · Train / valid / test summary

In [ ]:
# 6. Train / valid / test summary per model: mean over outer folds, whiskers = min-max.
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, metric in zip(axes, ["AUROC", "loss", "accuracy", "balanced_accuracy"]):
    for k, (split, colour) in enumerate(SPLIT_COLOURS.items()):
        g = split_metrics[split_metrics.split.eq(split)].groupby("model")[metric]
        mean, lo, hi = (g.agg(f).reindex(MODELS).to_numpy(float) for f in ["mean", "min", "max"])
        ok = np.isfinite(mean)
        x = np.arange(len(MODELS))[ok] + (k - 1) * 0.27
        ax.bar(
            x,
            mean[ok],
            0.27,
            yerr=[mean[ok] - lo[ok], hi[ok] - mean[ok]],
            capsize=2,
            color=colour,
            label=split,
        )
    ax.set_xticks(np.arange(len(MODELS)), MODELS, rotation=20, ha="right", fontsize=8)
    ax.set_title(
        {
            "AUROC": "AUROC",
            "loss": "class-balanced log loss (lower = better)",
            "accuracy": "accuracy at 0.5",
            "balanced_accuracy": "balanced accuracy at 0.5",
        }[metric],
        fontsize=9,
    )
axes[2].axhline((y == 0).mean(), color="grey", ls=":", lw=1, label="never fires")
axes[0].legend(fontsize=8)
axes[2].legend(fontsize=7)
save(fig, "06_split_summary")

### 7 · False alarms by deployment configuration, and the gate's ROC

In [ ]:
# 7. False alarms on held-out healthy participants by deployment configuration (high-specificity
#   points).
hs = false_alarms[
    false_alarms.subset.eq("all")
    & false_alarms.operating_point.isin(["definition", "high_specificity"])
]
config_rates = hs.pivot_table(index="model", columns="configuration", values="per_hour").reindex(
    MODELS
)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
config_rates.plot.barh(ax=axes[0])
axes[0].set(
    xlabel="false alert starts per monitored hour",
    title="Held-out WildPPG participants (high-specificity points)",
)
axes[0].legend(fontsize=7)
lab = transfer.ecg_beats.gt(0)
fpr, tpr, _ = roc_curve(transfer.visible[lab], transfer.gate_score[lab])
axes[1].plot(fpr, tpr, label=f"test participants (AUROC {gate_performance.AUROC.iloc[-1]:.2f})")
axes[1].plot([0, 1], [0, 1], ":", color="grey")
axes[1].set(
    xlabel="1 - specificity",
    ylabel="sensitivity",
    title="Pulse-visibility gate (PPG only; target: PPG beats agree with ECG)",
)
axes[1].legend(fontsize=8)
save(fig, "07_v5_gate_and_configurations")

## Export

In [ ]:
configuration = {
    "purpose": "pulse-lost / dangerous-rhythm detector from PPG",
    "version": "v5 (2026-11-16)",
    "deployable": False,
    "drowning_detector": False,
    "quick_smoke_test": QUICK,
    "run_plan_sha256": hashlib.sha256(_prereg.encode()).hexdigest(),
    "training_data": {
        "dataset": "PhysioNet/CinC Challenge 2015 training set",
        "md5": C2015[2],
        "labels": "expert true/false alarm annotations (Challenge)",
        "alarms_used": TARGET_ALARMS,
        "alarms_excluded": EXCLUDED_ALARMS,
        "records": int(len(y)),
        "true_events": int(y.sum()),
        "healthy_negatives": {
            "dataset": "WildPPG sternum IR PPG",
            "participants": WILDPPG_TRAIN_IDS,
            "windows": int(len(H_norm)),
            "label": "protocol fact: healthy volunteers",
            "selection": f"automatic ECG/PPG beat-count agreement ({VISIBLE_RATIO}), non-overlapping",
            "used_by": [m for m in MODELS if m.endswith("_h")],
        },
    },
    "false_alarm_data": {
        "dataset": "WildPPG sternum IR PPG, healthy volunteers",
        "participants": WILDPPG_TEST_IDS,
        "hours": float(len(transfer) * TRANSFER_STEP_S / 3600),
        "note": "disjoint from current 04 training/gate fitting; all appeared in prior 02 results; p5d used in ECG debugging; model selection uses this comparison",
    },
    "input": {
        "rate_hz": DEVICE_FS,
        "window_s": WINDOW_S,
        "band_hz": [0.5, 8.0],
        "scaling": "5-95 percentile spread",
        "ir_sign": "clinical PLETH convention (raw optical intensity negated)",
    },
    "rule_definitions": {
        "asystole_gap_s": ASYSTOLE_GAP_S,
        "brady_bpm": BRADY_BPM,
        "brady_beats": BRADY_BEATS,
    },
    "operating_points": {
        "high_sensitivity": TARGET_SENSITIVITY,
        "high_specificity": TARGET_SPECIFICITY,
    },
    "final_thresholds": final_thresholds,
    "features": FEATURES,
    "models": MODELS,
    "seed": SEED,
    "pulse_visibility_gate": {
        "features": GATE_FEATURES,
        "threshold": GATE_THRESHOLD,
        "threshold_rule": f"lowest with leave-one-participant-out precision >= {GATE_MIN_PRECISION}",
        "target": "PPG beats / ECG beats within 0.8-1.2 (reference-instrument agreement)",
        "applied_to": "the 16-s window ending where the scored window starts",
        "performance": gate_performance.to_dict("records"),
    },
    "persistence": {
        "k": PERSIST_K,
        "n": PERSIST_N,
        "role": "reported only",
        "combined_gate": "votes on detector flags, then apply the current prior-window gate",
    },
    "firmware_selection": selection,
    "training": {
        "loss": "class-weighted binary cross-entropy (balanced classes)",
        "cnn_1d": {
            "epochs": CNN4_EPOCHS,
            "batch": 32,
            "optimizer": "Adam lr 1e-3, weight decay 1e-3",
        },
        "cnn_v5": PREREG["cnn_v5"],
        "augmentation": PREREG["augmentation"],
        "boosted_trees": {"rounds": 200, "learning_rate": 0.05},
    },
    "limitations": [
        "ICU finger PPG, not chest MAX30102 in water",
        "record-level folds (no patient IDs published)",
        "few true asystole (17) and VF (6) events with PPG",
        "Poisson rate intervals assume independent starts; participant clustering can widen uncertainty",
        "all WildPPG participants inspected previously; selection comparison is not independent validation",
        "healthy-PPG test on land only",
        "gate target uses an automatic ECG beat count (reference instrument), not annotated beats",
        "gate transfer to ICU finger PPG is indicative only",
    ],
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "torch": torch.__version__ if HAVE_TORCH else None,
    },
}
(OUT / "pulse_loss_configuration.json").write_text(
    json.dumps(configuration, indent=2, default=str), encoding="utf-8"
)
(OUT / "firmware_selection.json").write_text(
    json.dumps(selection, indent=2, default=str), encoding="utf-8"
)
results.to_csv(OUT / "cv_results.csv", index=False)
per_type.to_csv(OUT / "cv_per_alarm_type.csv", index=False)
chosen.to_csv(OUT / "fold_thresholds.csv", index=False)
false_alarms.to_csv(OUT / "healthy_false_alarms.csv", index=False)
split_metrics.to_csv(OUT / "split_metrics.csv", index=False)
curves.to_csv(OUT / "training_curves.csv", index=False)
persistence_icu.to_csv(OUT / "persistence_icu.csv", index=False)
icu_gate_pass.to_csv(OUT / "icu_gate_pass.csv")
gate_performance.to_csv(OUT / "gate_performance.csv", index=False)
polarity_check.to_csv(OUT / "polarity_check.csv", index=False)
transfer.drop(columns=["motion"]).to_csv(OUT / "healthy_test_windows.csv.gz", index=False)
pd.DataFrame(
    {
        "record": meta.record,
        "alarm": meta.alarm,
        "label": y,
        "fold": fold_of,
        **{f"score_{m}": oof[m] for m in MODELS},
    }
).to_csv(OUT / "oof_predictions.csv", index=False)
joblib.dump(
    {m: v for m, v in final_models.items() if not m.startswith("cnn")},
    OUT / "pulse_loss_feature_models.joblib",
)
joblib.dump(
    {"model": gate_model, "threshold": GATE_THRESHOLD, "features": GATE_FEATURES},
    OUT / "pulse_visibility_gate.joblib",
)
if "cnn_1d" in final_models:
    torch.save(final_models["cnn_1d"].net.state_dict(), OUT / "pulse_loss_cnn_1d.pt")
for m in ("cnn_v5", "cnn_v5_h"):
    if m in final_models:
        torch.save(
            {
                "state_dicts": [n.state_dict() for n in final_models[m].nets],
                "best_epochs": final_models[m].best_epochs,
            },
            OUT / f"pulse_loss_{m}.pt",
        )
print("Exported:", sorted(p.name for p in OUT.iterdir() if p.is_file()))

## Reading the results

- **Start with `rule_definitions`.** It uses the Challenge's own published thresholds, so nothing about it was tuned. A trained model must beat it on *both* sensitivity and healthy false alarms to justify itself.
- **Check the train / valid / test gap before trusting a learned model.** If train AUROC is near 1 while valid and test are much lower, the model has memorised these 411 records. Valid close to test means the thresholds chosen on valid should hold on new records *from the same ICU population*. It says nothing about the chest in water.
- **Read accuracy with its baseline.** Never firing scores 0.76, so balanced accuracy and sensitivity are the informative numbers.
- **Read intervals, not points.** There are only 17 true asystole and 6 true VF events with PPG, so per-type results for those are anecdotal.
- **The two halves answer different questions.**
  - Cross-validation asks: can PPG tell a real dangerous rhythm from a false monitor alarm?
  - The WildPPG comparison measures alarm burden on the inspected healthy chest recordings.

  A deployable detector needs both, measured on the wearable's own signal in water.
- **Next data:** the wearable's MAX30102 recorded alongside a chest ECG during swimming (false alarms), and HealthRing or WildPPG motion segments for artefact robustness.
The inherited ranking filters on **ungated** ICU sensitivity and then ranks **gated** healthy alarm starts per monitored hour. It does not establish an operating policy that preserves sensitivity. Consult the gated sensitivity, monitored fraction, and flagged fraction before interpreting the selected model. The 90% gate precision target describes empirical agreement with another automatic estimate, not a clinical assurance.